# GinzburgLandau FEniCS Benchmark

This notebook demonstrates the current maintained CLI commands on lightweight benchmark cases.

Relevant docs:
- [docs/problems/GinzburgLandau.md](../../docs/problems/GinzburgLandau.md)
- [docs/results/GinzburgLandau.md](../../docs/results/GinzburgLandau.md)


In [1]:
from pathlib import Path
import json
import subprocess

REPO_ROOT = Path.cwd()
ARTIFACT_ROOT = Path('artifacts/raw_results/notebook_runs')
(REPO_ROOT / ARTIFACT_ROOT).mkdir(parents=True, exist_ok=True)
PYTHON = './.venv/bin/python'

def dump_json(path):
    with open(REPO_ROOT / path, encoding='utf-8') as handle:
        return json.load(handle)


In [2]:
from pprint import pprint

out_dir = ARTIFACT_ROOT / 'ginzburg_landau_fenics_benchmark'
(REPO_ROOT / out_dir).mkdir(parents=True, exist_ok=True)

def run_case(name, argv):
    out_path = out_dir / f'{name}.json'
    cmd = argv + [str(out_path)]
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)
    payload = dump_json(out_path)
    row = payload['results'][0]
    return {
        'case': name,
        'dofs': row.get('dofs', row.get('total_dofs')),
        'iters': row.get('iters'),
        'energy': row.get('energy'),
        'time': row.get('time', row.get('solve_time', row.get('total_time'))),
    }

results = [
    run_case('custom_serial', [PYTHON, 'src/problems/ginzburg_landau/fenics/solve_GL_custom_jaxversion.py', '--levels', '5', '--quiet', '--json']),
    run_case('snes_serial', [PYTHON, 'src/problems/ginzburg_landau/fenics/solve_GL_snes_newton.py', '--levels', '5', '--json']),
    run_case('custom_mpi2', ['mpiexec', '-n', '2', PYTHON, 'src/problems/ginzburg_landau/fenics/solve_GL_custom_jaxversion.py', '--levels', '5', '--quiet', '--json']),
]
pprint(results)


$ ./.venv/bin/python src/problems/ginzburg_landau/fenics/solve_GL_custom_jaxversion.py --levels 5 --quiet --json artifacts/raw_results/notebook_runs/ginzburg_landau_fenics_benchmark/custom_serial.json


Ginzburg-Landau 2D Custom Newton | 1 MPI process(es)
  --- Mesh level 5 ---
  RESULT mesh_level=5 dofs=4225 setup=0.018s solve=0.107s iters=8 ksp=33 asm=0.013s J(u)=0.346232 [Converged (energy, step, gradient)]
Done.
Results saved to artifacts/raw_results/notebook_runs/ginzburg_landau_fenics_benchmark/custom_serial.json


$ ./.venv/bin/python src/problems/ginzburg_landau/fenics/solve_GL_snes_newton.py --levels 5 --json artifacts/raw_results/notebook_runs/ginzburg_landau_fenics_benchmark/snes_serial.json


Ginzburg-Landau 2D SNES Newton | 1 MPI process(es)
  mesh_level=5 dofs=4225 time=0.036s iters=10 J(u)=0.3462 reason=2
Done.
Results saved to artifacts/raw_results/notebook_runs/ginzburg_landau_fenics_benchmark/snes_serial.json


$ mpiexec -n 2 ./.venv/bin/python src/problems/ginzburg_landau/fenics/solve_GL_custom_jaxversion.py --levels 5 --quiet --json artifacts/raw_results/notebook_runs/ginzburg_landau_fenics_benchmark/custom_mpi2.json


Ginzburg-Landau 2D Custom Newton | 2 MPI process(es)
  --- Mesh level 5 ---
  RESULT mesh_level=5 dofs=4225 setup=0.024s solve=0.079s iters=7 ksp=28 asm=0.007s J(u)=0.346232 [Converged (energy, step, gradient)]
Done.
Results saved to artifacts/raw_results/notebook_runs/ginzburg_landau_fenics_benchmark/custom_mpi2.json


[{'case': 'custom_serial',
  'dofs': 4225,
  'energy': 0.3462323651,
  'iters': 8,
  'time': 0.1067},
 {'case': 'snes_serial',
  'dofs': 4225,
  'energy': 0.3462,
  'iters': 10,
  'time': 0.0357},
 {'case': 'custom_mpi2',
  'dofs': 4225,
  'energy': 0.3462323651,
  'iters': 7,
  'time': 0.0789}]
